In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assuming we have a function `train_model` that trains the model
# and a function `evaluate_model` that evaluates the model's performance

def refactored_training_loop(data, model, epochs=100, batch_size=32):
    # Split the data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(data['features'], data['target'], test_size=0.2, random_state=42)

    for epoch in range(epochs):
        # Shuffle the training data
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train = X_train[indices]
        y_train = y_train[indices]

        # Train in batches
        for i in range(0, X_train.shape[0], batch_size):
            X_batch = X_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]
            train_model(model, X_batch, y_batch)

        # Evaluate the model on the validation set
        val_loss, val_accuracy = evaluate_model(model, X_val, y_val)
        print(f'Epoch {epoch + 1}/{epochs}, Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')

# Example usage
# refactored_training_loop(data, model)

In [3]:
from bayes_opt import BayesianOptimization

# Define the hyper-parameter tuning function using Bayesian optimization
def bayesian_optimization_tuning(data, model):
    def target_function(learning_rate, num_layers):
        # Set hyperparameters
        model.learning_rate = learning_rate
        model.num_layers = int(num_layers)

        # Train the model
        model.train(data)

        # Return the validation accuracy as the target to maximize
        return model.evaluate(data)

    # Define the parameter space for Bayesian optimization
    pbounds = {'learning_rate': (0.001, 0.1), 'num_layers': (1, 5)}

    # Initialize BayesianOptimization object
    optimizer = BayesianOptimization(f=target_function, pbounds=pbounds, random_state=42)

    # Perform Bayesian optimization
    optimizer.maximize(init_points=5, n_iter=10)

# Example usage
# bayesian_optimization_tuning(data, model)

ModuleNotFoundError: No module named 'bayes_opt'

In [4]:
from metagpt.tools.libs.terminal import Terminal
terminal = Terminal()
await terminal.run_command('pip install bayesian-optimization')

2025-09-14 13:08:16.395 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/mas-metagpt/Desktop/MetaGPT
2025-09-14 13:08:16.401 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/mas-metagpt/Desktop/MetaGPT
2025-09-14 13:08:22.615 | INFO     | metagpt.tools.libs.terminal:_check_state:55 - The terminal is at:


'Collecting bayesian-optimization\n  Downloading bayesian_optimization-3.1.0-py3-none-any.whl.metadata (11 kB)\nCollecting colorama>=0.4.6 (from bayesian-optimization)\n  Downloading colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)\nRequirement already satisfied: numpy>=1.25 in /home/mas-metagpt/anaconda3/envs/metagpt-3.10/lib/python3.10/site-packages (from bayesian-optimization) (1.26.4)\nRequirement already satisfied: scikit-learn>=1.0.0 in /home/mas-metagpt/anaconda3/envs/metagpt-3.10/lib/python3.10/site-packages (from bayesian-optimization) (1.3.2)\nRequirement already satisfied: scipy>=1.0.0 in /home/mas-metagpt/anaconda3/envs/metagpt-3.10/lib/python3.10/site-packages (from bayesian-optimization) (1.15.3)\nRequirement already satisfied: joblib>=1.1.1 in /home/mas-metagpt/anaconda3/envs/metagpt-3.10/lib/python3.10/site-packages (from scikit-learn>=1.0.0->bayesian-optimization) (1.5.1)\nRequirement already satisfied: threadpoolctl>=2.0.0 in /home/mas-metagpt/anaconda3/envs/metag

In [5]:
from bayes_opt import BayesianOptimization

def optimize_hyperparameters(data, model):
    # Define the function to optimize
    def model_evaluation(learning_rate, n_estimators):
        # Set the hyperparameters
        model.set_params(learning_rate=learning_rate, n_estimators=int(n_estimators))
        
        # Train and evaluate the model
        refactored_training_loop(data, model, epochs=10)  # Use a smaller number of epochs for optimization
        val_loss, val_accuracy = evaluate_model(model, data['val_features'], data['val_target'])
        
        # Return the negative accuracy as we want to maximize it
        return val_accuracy

    # Define the bounds for the hyperparameters
    pbounds = {
        'learning_rate': (0.01, 0.1),
        'n_estimators': (50, 200)
    }

    # Create the Bayesian optimizer
    optimizer = BayesianOptimization(
        f=model_evaluation,
        pbounds=pbounds,
        random_state=42,
    )

    # Optimize the hyperparameters
    optimizer.maximize(init_points=5, n_iter=15)

# Example usage
# optimize_hyperparameters(data, model)